# 미션 1: 생수 충전 라인 데이터
- 상황: 4,800건의 생산 기록에서 재검이 왜 나는지 살펴본다
- 목표: 오늘 배운 것(크기 확인 · 정렬 · 그룹 · 빈칸 · 관리선 · 진단표)을 한 바퀴 돈다

## Q1. 파일 열고 크기 확인하기

In [1]:
# 표를 다루는 도구를 pd라는 짧은 이름으로 불러온다
import pandas as pd

# 노트북은 day01/mission01_bottling 폴더에, 파일은 맨 위 data 폴더에 있으므로
# 두 칸 위로 올라갔다가(../..) data로 들어간다
df = pd.read_csv("../../data/day01_bottling.csv")

print("크기:", df.shape)

크기: (4800, 13)


4,800건이다. 눈으로 훑기엔 이미 많다. 하지만 열은 13개라 **열은 아직 눈으로 볼 수 있다.**<br>
공정 데이터는 열이 592개라 그것조차 안 됐다. 행이 많은 건 컴퓨터가 세어주니 괜찮고, 문제는 늘 열이다.

## Q2. 검사 결과 살펴보기

In [2]:
print(df["result"].value_counts())

# normalize=True — 개수 대신 비율로
print((df["result"].value_counts(normalize=True) * 100).round(2))

result
합격    4554
재검     246
Name: count, dtype: int64
result
합격    94.88
재검     5.12
Name: proportion, dtype: float64


재검이 **5.12%**다. 공정 데이터의 불량 6.6%와 비슷한 수준이다.<br>
맞히려는 쪽이 아주 적다는 구조가 똑같다. 스무 건에 한 건꼴.

## Q3. 제품 규격별로 몇 건씩인가

In [3]:
df["product"].value_counts()

product
500mL    2428
1L       1397
2L        975
Name: count, dtype: int64

[치우침 한 줄]<br>
500mL가 전체의 절반이고 2L는 20%뿐이다. 규격별로 비교할 때<br>
2L 쪽은 건수가 적어 평균이 덜 안정적일 수 있다.

## Q4. 유독 뜨겁게 밀봉한 묶음 찾기

In [4]:
# ascending=False — 큰 값이 위로 오게(내림차순)
df.sort_values("seal_temp", ascending=False).head(5)[
    ["lot_id", "line_id", "seal_temp", "result"]
]

,lot_id,line_id,seal_temp,result
4554,L04555,F-2,209.5,재검
3946,L03947,F-2,207.2,재검
4031,L04032,F-2,207.0,재검
4703,L04704,F-2,205.9,재검
4383,L04384,F-2,205.5,재검


[공통점 한 줄]<br>
상위 5건이 전부 F-2 라인이다. 그리고 전부 재검이다.<br>
<br>
온도가 높은 게 흩어져 있는 게 아니라 한 라인에 몰려 있다. 설비 하나에 문제가 있다는 뜻일 수 있다.<br>
정렬 한 번으로 이만큼 나온다 — 열이 적을 때 정렬은 가장 값싼 발견 도구다.

## Q5. 라인별로 재검률이 다른가

In [5]:
# result == "재검" 이면 참(1), 아니면 거짓(0)
# 그것의 평균을 내면 곧 재검 비율이 된다
(df.groupby("line_id")["result"].apply(lambda s: (s == "재검").mean()) * 100).round(2)

line_id
F-1    4.65
F-2    5.29
F-3    5.65
Name: result, dtype: float64

### 여기서 조심할 것

Q4에서 "F-2가 문제"라는 인상을 받았는데, **재검률만 보면 F-3이 더 높다.** F-2는 중간이다.<br>
F-2에서 눈에 띈 건 온도가 극단적으로 튄 소수 건이었고, 그건 F-2 전체 재검률을 크게 올릴 만큼 많지 않았다.<br>
게다가 세 라인의 차이(4.65 ~ 5.65)가 의미 있는 차이인지도 아직 모른다.<br>
**"F-3이 나쁘다"고 쓰면 안 되고, "F-3에서 다소 높게 관찰됨"까지가 맞다.**

## Q6. 비어 있는 칸 찾기

In [6]:
# 열마다 비어 있는 칸 개수
print(df.isna().sum())

# 빈칸이 하나라도 있는 열이 몇 개인지
print("빈칸 있는 열 개수:", (df.isna().sum() > 0).sum())

lot_id                0
produced_at           0
plant_code            0
line_id               0
shift                 0
product               0
fill_error            0
cap_torque          139
seal_temp             0
line_speed            0
ambient_temp          0
ambient_humidity     74
result                0
dtype: int64
빈칸 있는 열 개수: 2


빈칸이 있는 두 열(cap_torque, ambient_humidity)은 다 **센서에서 자동으로 들어오는 값**이다.<br>
사람이 적는 값(lot_id·product)에는 빈칸이 없다. 센서 통신이 끊기면 그 칸이 빈다.<br>
공정 데이터에서는 590열 중 538열(91%)에 빈칸이 있었다. 여기는 13열 중 2열. 규모가 다를 뿐 성격은 같다.

## Q7. 캡 토크로 거르면 몇 건이 빠질까

In [7]:
print("전체:", len(df))
print("2.0 초과:", len(df[df["cap_torque"] > 2.0]))
print("2.0 이하:", len(df[df["cap_torque"] <= 2.0]))
print("합:", len(df[df["cap_torque"] > 2.0]) + len(df[df["cap_torque"] <= 2.0]))

전체: 4800
2.0 초과: 3340
2.0 이하: 1321
합: 4661


### 왜 그런가

**비어 있는 칸은 어느 비교에서도 참이 되지 않기 때문이다.**<br>
Q6에서 확인한 cap_torque 빈칸 139건이 `> 2.0`에서도 거짓, `<= 2.0`에서도 거짓이라<br>
양쪽 어디에도 안 잡힌다. 4,800 − 139 = 4,661.<br>
<br>
오류도 안 나고 경고도 안 뜬다. "토크 미달이 1,321건입니다"라고 보고했는데,<br>
실은 판단조차 못 한 139건이 따로 있었던 것이다.<br>
**막는 방법은 하나 — 거른 결과의 합이 전체와 맞는지 확인하는 것.**

## Q8. 교대조에 따라 다른가

In [8]:
df.groupby("shift")["cap_torque"].mean().round(3)

shift
야간    2.040
오후    2.084
주간    2.090
Name: cap_torque, dtype: float64

[한 줄]<br>
야간조의 캡 토크 평균이 다른 조보다 약간 낮게 관찰된다.<br>
다만 차이가 0.05 수준이라 이것만으로 원인을 말하기는 어렵다.<br>
<br>
⚠️ **"야간조가 일을 대충 한다" 같은 해석은 절대 쓰면 안 된다.** 데이터에 사람 이야기는 없다.<br>
데이터로 말할 수 있는 건 "이 조에서 값이 낮게 관찰됨"까지다.

## Q9. 밀봉 온도에 관리선 긋기 ⭐

### 밀봉 온도 관리선
평균에서 표준편차 3배만큼 위아래로 선을 긋고 벗어난 묶음을 찾는다.

In [9]:
평균 = df["seal_temp"].mean()
표준편차 = df["seal_temp"].std()

위선 = 평균 + 3 * 표준편차
아래선 = 평균 - 3 * 표준편차

print("평균:", round(평균, 2))
print("관리 상한:", round(위선, 2))
print("관리 하한:", round(아래선, 2))

# 위로 넘었거나 아래로 넘은 묶음만 고른다
벗어남 = df[(df["seal_temp"] > 위선) | (df["seal_temp"] < 아래선)]

print()
print("관리선 벗어난 건수:", len(벗어남))
print(벗어남["result"].value_counts())
print()
print("전체 재검:", (df["result"] == "재검").sum())

평균: 180.35
관리 상한: 188.3
관리 하한: 172.39

관리선 벗어난 건수: 52
result
합격    35
재검    17
Name: count, dtype: int64

전체 재검: 246


### 세 숫자 정리

| | 건수 |
|---|---|
| 관리선을 벗어난 묶음 | **52건** |
| 그중 실제 재검 | **17건** |
| 그중 실제로는 합격 | **35건** |
| 전체 재검 | **246건** |
| 관리도가 못 잡은 재검 | **229건** |

**① 관리선에 걸렸지만 합격이었던 35건은 현장에서 무슨 일이 벌어지나?**<br>
품질팀이 35번 확인하러 갔는데 35번 다 멀쩡했다는 뜻이다.<br>
헛걸음이다. 이런 게 계속되면 현장에서 경보를 안 믿게 된다.<br>
<br>
**② 재검인데 관리선에 안 걸린 229건은 무엇을 뜻하나?** ⭐<br>
밀봉 온도만으로는 재검의 대부분을 설명하지 못한다는 뜻이다.<br>
관리도가 잡은 건 246건 중 17건, 7%도 안 된다.<br>
나머지는 토크나 충전 오차 등 다른 값이 문제였을 것이다.

In [10]:
# 재검인데 밀봉 온도는 정상이었던 묶음들
숨은재검 = df[(df["result"] == "재검") & (df["seal_temp"] <= 위선)]

print("온도로 못 잡은 재검:", len(숨은재검))
print(숨은재검[["cap_torque", "fill_error"]].describe().round(2))

온도로 못 잡은 재검: 229
       cap_torque  fill_error
count      220.00      229.00
mean         1.76       -0.09
std          0.36        3.02
min          1.21       -5.49
25%          1.41       -1.75
50%          1.88        0.02
75%          2.08        1.40
max          2.41        5.50


> ⚠️ 위 코드는 **위쪽 선만** 본다. 관리 하한보다 낮아서 걸린 재검이 있으면 이중으로 세어진다.<br>
> 이 데이터에는 하한 아래 재검이 없어서 우연히 맞아떨어졌다. 아래처럼 두 선을 다 보는 편이 안전하다.

In [11]:
# 두 선을 모두 보고 고른 경우 — 위와 개수가 같은지 확인한다
숨은재검_안전 = df[(df["result"] == "재검") &
                (df["seal_temp"] <= 위선) &
                (df["seal_temp"] >= 아래선)]

print("위쪽 선만 본 경우:", len(숨은재검))
print("두 선을 다 본 경우:", len(숨은재검_안전))
print("전체 재검 - 관리선에 걸린 재검:", (df["result"] == "재검").sum() - (벗어남["result"] == "재검").sum())

위쪽 선만 본 경우: 229
두 선을 다 본 경우: 229
전체 재검 - 관리선에 걸린 재검: 229


### 이 미션에서 가장 중요한 발견

관리도는 **틀리지 않았다.** 온도가 튄 구간을 정확히 찾아냈고, 그중 17건은 실제 재검이었다.<br>
**문제는 두 방향으로 동시에 생긴다.**<br>
· 잡은 52건 중 **35건은 헛걸음** → 확인하러 갔는데 멀쩡했다<br>
· 재검 246건 중 **229건은 그냥 지나감** → 온도가 멀쩡해서 경보가 안 울렸다<br>
<br>
관리선을 좁히면 놓치는 건 줄지만 헛걸음이 훨씬 는다. 넓히면 반대다. **한쪽을 좋게 하면 다른 쪽이 나빠진다.**<br>
그리고 더 근본적인 문제 — **"온도가 조금 높은데 토크도 조금 낮은" 묶음**은 각각 따로 보면 어느 선도 안 넘는다.<br>
조합이 문제인 건데 한 열씩 보는 방식으로는 절대 안 잡힌다.<br>
여기는 열이 13개라 하나씩 선을 그어볼 수도 있다. **그런데 590개면?**

## Q10. 열마다 한 줄로 요약하기

In [12]:
숫자열 = ["fill_error", "cap_torque", "seal_temp",
          "line_speed", "ambient_temp", "ambient_humidity"]

진단표 = pd.DataFrame({
    "빈칸비율(%)": (df[숫자열].isna().sum() / len(df) * 100).round(2),
    "값종류수": df[숫자열].nunique(),
    "표준편차": df[숫자열].std().round(3),
    "최솟값": df[숫자열].min(),
    "최댓값": df[숫자열].max(),
})

진단표

,빈칸비율(%),값종류수,표준편차,최솟값,최댓값
fill_error,0.00,570,1.093,-5.49,5.50
cap_torque,2.90,124,0.165,1.21,2.58
seal_temp,0.00,183,2.652,173.40,209.50
line_speed,0.00,81,12.048,351.00,442.00
ambient_temp,0.00,193,3.235,14.30,36.30
ambient_humidity,1.54,423,7.918,22.40,82.50


### 쓸모없는 열

**plant_code 열이다.** 모든 기록이 같은 공장(P-SEOUL-01)에서 나와서 값이 한 종류뿐이다.

In [13]:
df["plant_code"].value_counts()

plant_code
P-SEOUL-01    4800
Name: count, dtype: int64

합격이든 재검이든 **전부 같은 값**이라, 둘을 구분하는 데 아무 기여를 못 한다. 실습 5에서 걸러낸 **상수열**이 이것이다.<br>
<br>
숫자 열만 진단표에 넣으면 plant_code는 아예 보이지도 않는다. **글자 열에도 상수열이 있을 수 있다.**<br>
그래서 전체 열을 훑어봐야 한다 — 아래처럼.

In [14]:
# 글자 열까지 포함해 값 종류가 1개인 열을 전부 찾는다
for c in df.columns:
    if df[c].nunique() == 1:
        print("상수열:", c, "->", df[c].dropna().unique())

상수열: plant_code -> <ArrowStringArray>
['P-SEOUL-01']
Length: 1, dtype: str


---
## 이 미션이 확인한 것

| 문항 | 오늘 배운 것 중 | 어디서 |
|---|---|---|
| Q1~Q3 | 크기·값 세기·그룹 크기 확인 | 실습 1 빈칸 Q1·Q2·Q5 |
| Q4 | 정렬로 극단값 찾기 | 실습 1 빈칸 Q4 |
| Q5 | 그룹으로 묶어 계산 · 인상과 숫자가 다를 때 | 실습 1 빈칸 Q9 |
| Q6~Q7 | 빈칸 세기 + 빈칸이 조용히 빠지는 함정 | 실습 1 도전 |
| Q8 | 그룹 비교 · 사람 이야기로 해석하지 않기 | 신규 |
| Q9 | 관리선 · 헛걸음과 놓친 것 | 실습 4 |
| Q10 | 진단표 · 상수열 찾기 | 실습 5 |

### 그리고 하나 더

이 생수 라인 데이터와 반도체 공정 데이터는 **구조가 똑같다.**

| | 생수 충전 라인 | 반도체 공정 |
|---|---|---|
| 크기 | 4,800행 × 13열 | 1,567행 × 592열 |
| 한 줄 = | 한 묶음의 생산 기록 | 한 번의 생산 흐름 |
| 조건 열 | 충전 오차·토크·밀봉 온도·속도·온습도 | 센서 590개 |
| 결과 열 | 합격 / 재검 | 양품 / 불량 |
| 소수인 쪽 | 재검 5.1% | 불량 6.6% |
| 빈칸 | 2개 열 | 538개 열 |
| 상수열 | 1개 (plant_code) | 116개 |
| 한 열씩 보면 | 재검 246건 중 17건만 잡힘 | 마찬가지 |

소재만 다르지 **푸는 방식은 같다.**